# 🧠 GYAN Analytics — LLM Fine-Tuning Pipeline (Version 2)
**Team:** MPR_CS_26_T7 | Gagan G, Esha Kumara, Bharathkumar G, Atharva Joshi

This notebook fine-tunes **Qwen2.5-0.5B-Instruct** using QLoRA.
It saves the new weights into a separate `gyan_analytics_model_v2` folder so your previous model is **100% preserved and never overwritten**.

- **Base Architecture:** `Qwen/Qwen2.5-0.5B-Instruct`
- **Method:** QLoRA (4-bit Quantization via BitsAndBytes + PEFT LoRA adapters)
- **Hardware:** Google Colab T4 GPU (Free tier)
- **Training Time:** ~6 to 8 minutes

**Instructions:**
1. Go to `Runtime` -> `Change runtime type` -> Select **T4 GPU**.
2. Run all cells sequentially from top to bottom.

## ⚡ Step 1 — Verify GPU & Install Dependencies

In [ ]:
# Check GPU availability
!nvidia-smi

# Install and upgrade libraries with compatible versions
!pip install -q --upgrade pip
!pip install -q --upgrade transformers datasets peft trl accelerate bitsandbytes
# Prevent torchao compatibility issues
!pip install -q --upgrade torchao || true

import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')
print('Required ML libraries installed successfully!')

## 📂 Step 2 — Mount Google Drive & Locate Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

DATA_DIR   = '/content/drive/MyDrive/GYAN_TRAINING_DATA/output'
TRAIN_FILE = os.path.join(DATA_DIR, 'gyan_train.jsonl')
VAL_FILE   = os.path.join(DATA_DIR, 'gyan_val.jsonl')

# Saved to v2 so your previous hours of training are 100% preserved!
MODEL_OUT  = '/content/drive/MyDrive/GYAN_TRAINING_DATA/gyan_analytics_model_v2'
os.makedirs(MODEL_OUT, exist_ok=True)

assert os.path.exists(TRAIN_FILE), f'Train file not found at {TRAIN_FILE}.'
assert os.path.exists(VAL_FILE), f'Val file not found at {VAL_FILE}.'

train_mb = os.path.getsize(TRAIN_FILE) / (1024 * 1024)
val_mb   = os.path.getsize(VAL_FILE) / (1024 * 1024)
print(f'Found Training data: {TRAIN_FILE} ({train_mb:.1f} MB)')
print(f'Found Validation data: {VAL_FILE} ({val_mb:.1f} MB)')
print(f'Target output directory (v2): {MODEL_OUT}')

## 🔄 Step 3 — Format Dataset into Chat Conversations

In [ ]:
from datasets import load_dataset

SYSTEM_PROMPT = (
    "You are an expert data analyst and business intelligence specialist for GYAN Analytics. "
    "Your task is to analyze statistical patterns, anomalies, customer/data segments, correlations, "
    "and time-series trends, and generate a clear, professional, multi-step analytical insight report "
    "with root causes and actionable recommendations."
)
raw_datasets = load_dataset('json', data_files={'train': TRAIN_FILE, 'val': VAL_FILE})
print('Raw Dataset Sample Counts:', raw_datasets)

def format_to_chat(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Analyze the following dataset findings and provide a comprehensive insight report:\n\n{example['input']}"},
        {"role": "assistant", "content": example['output']}
    ]
    return {"messages": messages}

chat_dataset = raw_datasets.map(format_to_chat, remove_columns=raw_datasets['train'].column_names)
print('\nConverted to Chat format. First sample preview:')
print('System:', chat_dataset['train'][0]['messages'][0]['content'][:100], '...')
print('User:', chat_dataset['train'][0]['messages'][1]['content'][:140], '...')
print('Assistant:', chat_dataset['train'][0]['messages'][2]['content'][:140], '...')

## 🤖 Step 4 — Load Pretrained Base Model with 4-Bit QLoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
print(f'Loading base model: {BASE_MODEL_NAME} in 4-bit...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('QLoRA adapters attached successfully!')

## 🏋️ Step 5 — Configure & Launch SFTTrainer (~6 Minutes)

In [ ]:
from trl import SFTTrainer, SFTConfig

# Select 2,500 training examples for optimal convergence in ~6 minutes on Colab T4
num_train = min(2500, len(chat_dataset['train']))
num_val   = min(250, len(chat_dataset['val']))

train_sample = chat_dataset['train'].shuffle(seed=42).select(range(num_train))
val_sample   = chat_dataset['val'].shuffle(seed=42).select(range(num_val))

# Apply chat template to generate training text strings
def apply_template_to_example(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False
    )
    return {'text': text}

train_formatted = train_sample.map(apply_template_to_example)
val_formatted   = val_sample.map(apply_template_to_example)

training_args = SFTConfig(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    fp16=False,
    bf16=False,
    logging_steps=25,
    save_strategy='no',
    dataset_text_field='text',
    report_to='none'
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    args=training_args
)

print(f'Starting fine-tuning on {num_train} examples (~6 minutes on T4)...')
trainer.train()
print('Training complete!')

## 🧪 Step 6 — Interactive Test (Inference on a New Dataset)

In [ ]:
def test_model(test_input):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Analyze the following dataset findings and provide a comprehensive insight report:\n\n{test_input}"}
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=400,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()

# Test: Retail Sales dataset findings
sample_input = (
    "Domain: retail || Rows: 500 | Columns: 6 || Missing data: 0.2% | Duplicates: 0.0% || "
    "Column stats: revenue(mean=2450.0,std=1100.5,min=120.0,max=14500.0) | orders(mean=34,std=15,min=1,max=180) || "
    "Anomalies: 8 rows flagged (1.6% of data) || "
    "Clusters (k=3): High Spenders: 85 rows (17.0%) | Regulars: 310 rows (62.0%) | Occasional: 105 rows (21.0%) || "
    "Key correlations: orders ↔ revenue: r=0.89 (strong, positive) || "
    "Trends: revenue: upward trajectory (R²=0.72)"
)

print('INPUT FINDINGS:\n', sample_input)
print('\n' + '='*80 + '\n')
print('MODEL GENERATED INSIGHT REPORT:\n')
print(test_model(sample_input))
print('\n' + '='*80)

## 💾 Step 7 — Save the Fine-Tuned Model to Google Drive

In [ ]:
print(f'Saving LoRA adapter weights and tokenizer to: {MODEL_OUT}...')
trainer.model.save_pretrained(MODEL_OUT)
tokenizer.save_pretrained(MODEL_OUT)

print('LoRA weights and tokenizer saved successfully!')
print(f'Files in {MODEL_OUT}:')
for f in os.listdir(MODEL_OUT):
    f_size = os.path.getsize(os.path.join(MODEL_OUT, f))
    print(f'   📁 {f} ({f_size / 1024:.1f} KB)')

print('\nCongratulations! Your custom GYAN Analytics AI model v2 is fully trained and saved in Google Drive!')
print('Your original model is preserved, and the new model is ready in gyan_analytics_model_v2.')